<p style="text-align: center;">
  <img src="https://github.com/digital-futures-academy/DataScienceMasterResources/blob/main/Resources/datascience-notebook-header.png?raw=true"
       alt="DigitalFuturesLogo"
       style="display: inline-block; margin: 0 auto;" />
</p>

# Frontier AI: Pandas Essentials Workbook

(V1) This notebook is a **mini reference of core Pandas skills**, using a small toy dataset so you can:
- See the pattern
- Copy/paste
- Adapt to your own projects

You don't need any external files — the data is created inside the notebook.

### Contents
1. Setup & Toy Dataset  
2. Inspecting DataFrames  
3. Selecting Rows & Columns  
4. Filtering Data  
5. Common Transformations (rename, sort, drop, new columns)  
6. Handling Missing Data  
7. GroupBy & Aggregation  
8. Pivoting & Reshaping  
9. Working with Dates & Times  
10. Merging / Joining DataFrames  
11. Basic Plotting  


---
## 1) Setup & Toy Dataset

We'll create a **toy sales dataset** with columns like:

- `order_id` — unique order ID  
- `customer` — customer name  
- `region` — sales region  
- `product` — product name  
- `quantity` — number of items  
- `unit_price` — price per item  
- `order_date` — date of the order  

Run the cell below to set everything up.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = {
    "order_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "customer": [
        "Amelia", "Mohammed", "Olivia", "Arjun", "Grace",
        "Aaliyah", "James", "Zara", "Luca", "Patrick"
    ],
    "region": [
        "London", "Scotland", "Wales", "North West", "South East",
        "Yorkshire", "Midlands", "South West", "Northern Ireland", "London"
    ],
    "product": [
        "Laptop", "Wireless Mouse", "Mechanical Keyboard", "Laptop",
        "4K Monitor", "4K Monitor", "Wireless Mouse", "Laptop",
        "Mechanical Keyboard", "Wireless Mouse"
    ],
    "quantity": [1, 2, 1, 3, 2, 1, 5, 2, 4, 3],
    "unit_price": [1000, 25, 45, 1000, 300, 320, 25, 950, 40, 25],
    "order_date": [
        "2024-01-01", "2024-01-03", "2024-01-05", "2024-01-05", "2024-01-07",
        "2024-01-09", "2024-01-10", "2024-01-11", "2024-01-13", "2024-01-15"
    ]
}

df = pd.DataFrame(data)

# Convert order_date to datetime
df['order_date'] = pd.to_datetime(df['order_date'])

df

---
## 2) Inspecting DataFrames

Some of the **most important commands** when you first meet a dataset:

- `df.head()` — first 5 rows  
- `df.tail()` — last 5 rows  
- `df.shape` — number of rows and columns  
- `df.info()` — data types + non-null counts  
- `df.describe()` — summary statistics for numeric columns  
- `df.sample(n)` — random sample of rows  


Show the first 5 rows

In [ ]:
# EX
df.head()

Show the last 3 rows

In [ ]:
# EX
df.tail(3)

Inspect the shape (rows, columns) - Note that this one is an attribute not a method!

In [ ]:
# EX
df.shape

Find the info - Data types and non-null counts

In [ ]:
# EX
df.info()

Summary statistics for numeric columns

In [ ]:
# EX
df.describe()

**Practice:** Get summary statistics for ALL columns (try using describe but set the argument `include` to `"all"`)

In [ ]:
# EX
df.describe(include="all")

**Practice:** Get a random sample of 4 rows, use a `random_state` of 124

In [ ]:
# EX
df.sample(4, random_state=124)

---
## 3) Selecting Rows & Columns

### Selecting columns
- Single column: `df['column']` → Series  
- Multiple columns: `df[['col1', 'col2']]` → DataFrame  

### Selecting rows by position: `.iloc`
- First row: `df.iloc[0]`  
- First 3 rows: `df.iloc[:3]`  

### Selecting rows by label: `.loc`
- Row with index label 101: `df.loc[101]` (after we set an index)  

We'll also set `order_id` as the index to see how `.loc` works nicely.


**Practice:** Select the customer name column and display it

In [ ]:
# EX
customer_series = df['customer']
customer_series

**Practice:** Select customer, region and product and display the first 5 rows

In [ ]:
# EX
df[['customer', 'region', 'product']].head()

Select just row 0 with `.iloc[]` (returns a Series)

In [ ]:
# EX
df.iloc[0]

**Practice:** Select the first three rows with `.iloc[]` (returns a DataFrame)

In [ ]:
# EX
df.iloc[:3]

Set `order_id` as index so we can use `.loc`

In [ ]:
# EX
df = df.set_index('order_id')
df.head()

**Practice:** Select order 103 using `.loc`

In [ ]:
# EX
df.loc[103]

**Practice:** Select orders 101 to 104, and the customer, product and quantity columns

In [ ]:
# EX
df.loc[101:104, ['customer', 'product', 'quantity']]

---
## 4) Filtering Data

Filtering, or selecting certain rows is one of the **most common things** you'll do. It's similar to a where clause in SQL.

The way it works in Pandas is interesting - we use a Boolean mask.

Pattern:
```python
mask = df['column'] <condition>
df_filtered = df[mask]
```

We can combine conditions with:
- `&`  (AND)  
- `|`  (OR)  
- `~`  (NOT)  


Select all orders from the 'London' region

In [ ]:
# EX
london_orders = df[df['region'] == 'London']
london_orders

**Practice:** Select all of the Orders where quantity >= 3

In [ ]:
# EX
big_orders = df[df['quantity'] >= 3]
big_orders

**Practice:** Find any Orders from South East, South West, or London

In [ ]:
# EX
south_region = df[(df['region'] == 'South West') | (df['region'] == 'South East') | (df['region'] == 'London')]
south_region

**Practice:** Find any orders that are NOT from London

In [ ]:
# EX
not_london = df[df['region'] != 'London']
not_london

Using `isin` for multiple matches - find orders for "Wireless Mouse" or "Mechanical Keyboard"

In [ ]:
# EX
mousekeyboard = df[df['product'].isin(['Wireless Mouse', 'Mechanical Keyboard'])]
mousekeyboard

---
## 5) Common Transformations

Key patterns:

- **Creating new columns**
  ```python
  df['new_col'] = ...
  ```

- **Renaming columns**
  ```python
  df.rename(columns={'old': 'new'}, inplace=True)
  ```

- **Sorting**
  ```python
  df.sort_values('column', ascending=False)
  ```

- **Dropping columns**
  ```python
  df.drop(columns=['col1', 'col2'])
  ```


**Practice:** Create a `total_value` column = quantity * unit_price

In [ ]:
# EX
df['total_value'] = df['quantity'] * df['unit_price']
df[['customer', 'product', 'quantity', 'unit_price', 'total_value']].head()

Rename `region` → `sales_region`

In [ ]:
# EX
df = df.rename(columns={'region': 'sales_region'})
df.head()

**Practice:** Sort by `total_value`, biggest first

In [ ]:
# EX
df_sorted = df.sort_values('total_value', ascending=False)
df_sorted[['customer', 'product', 'total_value']].head()

**Practice:** Create a dataframe called `anon` that contains everything except the customer column

In [ ]:
# EX
anon = df.drop(columns=['customer'])
anon.head()

---
## 6) Handling Missing Data

Key methods:

- `df.isna()` — True/False for missing values  
- `df.isna().sum()` — count missing per column  
- `df.dropna()` — drop rows with missing values  
- `df.fillna(value)` — fill missing values  

We'll first create a small version with some missing values.


Create a small version with missing values

In [ ]:
# EX
df_na = df.copy()
df_na.loc[101, 'unit_price'] = np.nan
df_na.loc[105, 'customer'] = np.nan

df_na

Where are the missing values?

In [ ]:
# EX
df_na.isna()

**Practice:** Count missing values per column

In [ ]:
# EX
df_na.isna().sum()

**Practice:** Fill missing `unit_price` with the mean unit_price

In [ ]:
# EX
mean_price = df_na['unit_price'].mean()
df_filled = df_na.copy()
df_filled['unit_price'] = df_filled['unit_price'].fillna(mean_price)
df_filled

**Practice:** Drop rows with ANY missing values

In [ ]:
# EX
df_dropna = df_na.dropna()
df_dropna

---
## 7) GroupBy & Aggregation

Pattern:

```python
df.groupby('column')['other_column'].agg(['sum', 'mean', 'count'])
```

Useful for:
- Total sales per customer
- Average order value per region
- Number of orders per product


**Practice:** Find the total quantity per customer (i.e. sum of quantity per customer)

In [ ]:
# EX
qty_per_customer = df.groupby('customer')['quantity'].sum()
qty_per_customer

**Practice:** Find the total sales value per customer (i.e. sum the `total_value` column for each customer)

In [ ]:
# EX
sales_per_customer = df.groupby('customer')['total_value'].sum()
sales_per_customer

**Practice:** Multiple aggregations per customer - find the sum, mean and count of `total_value`

In [ ]:
# EX
customer_stats = df.groupby('customer')['total_value'].agg(['sum', 'mean', 'count'])
customer_stats

**Practice:** Group by two columns: `sales_region` and `product` and find the sum of `total_value`

In [ ]:
# EX
region_product_stats = df.groupby(['sales_region', 'product'])['total_value'].sum()
region_product_stats

---
## 8) Pivoting & Reshaping

### `pivot_table`
Turn "long" data into a table with rows and columns.

```python
pd.pivot_table(df, values='total_value', index='sales_region', columns='product', aggfunc='sum')
```

### `melt`
Go from wide → long.

```python
df_long = df.melt(id_vars=[...], value_vars=[...])
```


**Practice:** Create a pivot table of the data.
- The value should be `total_value`
- The index should be `sales_region`
- The columns should be `product`
- The aggfunc should be `sum`
- Fill any missing values with 0

In [ ]:
# EX
pivot_sales = pd.pivot_table(
    df,
    values='total_value',
    index='sales_region',
    columns='product',
    aggfunc='sum',
    fill_value=0
)
pivot_sales

---
# 9) Working with Dates & Times

We already converted `order_date` to datetime.

Useful patterns:

- Extract parts:
  ```python
  df['order_date'].dt.year
  df['order_date'].dt.month
  df['order_date'].dt.day_name()
  ```

- Resample (needs a DateTimeIndex):
  ```python
  df.set_index('order_date').resample('D').sum()
  ```


**Practice:** Extract year, month, day name into new columns called `year`, `month`, `day_name`

In [ ]:
# EX
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['day_name'] = df['order_date'].dt.day_name()

df[['order_date', 'year', 'month', 'day_name']].head()

**Practice:** Create a date index. Then create a new dataframe called `daily_sales` that is resampled to a daily frequency and sums the `total_value` column

In [ ]:
# EX
df_date_index = df.set_index('order_date')

daily_sales = df_date_index['total_value'].resample('D').sum()
daily_sales

---
## 10) Merging / Joining DataFrames

Very common pattern when data comes from multiple tables.

Pattern:

```python
pd.merge(left_df, right_df, on='key_column', how='left')
```

We'll create a small **customers table** and merge it with the orders.


Customers table with extra info

In [ ]:
# EX
customers = pd.DataFrame({
    'customer': ["Amelia", "Mohammed", "Olivia", "Arjun", "Grace",
        "Aaliyah", "James", "Zara", "Luca", "Patrick"],
    'loyalty_tier': ['Gold', 'Silver', 'Bronze', 'Gold', 'Silver', 'Bronze', 'Gold', 'Silver', 'Bronze', 'Gold']
})
customers

**Practice:** Merge the new customers table into the original df. Merge on `customer`, with a left join.

In [ ]:
# EX
merged = pd.merge(
    df.reset_index(),  # bring order_id back as a column
    customers,
    on='customer',
    how='left'   # keep all orders even if customer not in customers table
)
merged

# In the next workshops, We'll learn how to make graphs in Matplotlib and Seaborn!